# Step 3: Agents in a Workflow with Streaming

## Overview

This notebook demonstrates **streaming workflow execution**, allowing you to observe events as they occur in real-time. We'll build a two-agent workflow with:
1. **Writer Agent** - Generates content
2. **Reviewer Agent** - Reviews and finalizes the result

### Key Concepts:

- **Streaming Execution**: Using `workflow.run_stream()` to observe events in real-time
- **Custom Executor Classes**: Creating reusable executor components with embedded agents
- **Event Types**: Understanding different workflow events (status, output, errors)
- **Event Origin**: Distinguishing runner-generated vs executor-generated events
- **Typed Contexts**: Using `WorkflowContext[T_Out, T_W_Out]` for type safety

### Prerequisites:

- ✅ Microsoft Foundry Project configured with required environment variables
- ✅ Azure CLI authentication (`az login` completed)

## Import Required Libraries

In [ ]:
import asyncio

from agent_framework import (
    AgentRunEvent,
    WorkflowBuilder,
    ChatMessage,
    Executor,
    WorkflowContext,
    handler,
    WorkflowOutputEvent,
    WorkflowStatusEvent,
    WorkflowRunState,
    ExecutorFailedEvent,
    WorkflowFailedEvent,
)
from azure.identity.aio import AzureCliCredential
from agent_framework.azure import AzureAIAgentClient
from pathlib import Path  # For working with file paths
import os  # For environment variables
import time  # For sleep function
from dotenv import load_dotenv  # For loading environment variables from .env file
# Get the path to the .env file which is in the parent directory
notebook_path = Path().absolute()  # Get absolute path of current notebook
parent_dir = notebook_path.parent  # Get parent directory
load_dotenv('../../.env')  # Load environment variables from .env file

## Create Agent Framework Agent Client

We'll use **Microsoft Foundry Project** with `AzureCliCredential` for authentication.

### Environment Variables Required:
- **AI_FOUNDRY_PROJECT_ENDPOINT**: Your Azure OpenAI endpoint URL
- **MODEL_DEPLOYMENT_NAME**: Your model deployment name (e.g., "gpt-4o")

These should be set in the `.env` file located at `python/samples/getting_started/.env`

In [ ]:
# Create the Azure AI credential
credential = AzureCliCredential()

print("✅ Azure AI credential created successfully!")

## Define the Writer Executor

### Custom Executor Pattern

This class demonstrates:
- Attaching a `ChatAgent` to an `Executor` for workflow participation
- Using `@handler` method with typed input and output
- Forwarding typed output via `ctx.send_message()`

### Handler Contract:
- **Input**: `ChatMessage` (inbound user message)
- **Context**: `WorkflowContext[list[ChatMessage]]` (expects list of messages downstream)

### Pattern:
1. Seed conversation with inbound message
2. Run attached agent to produce assistant messages
3. Forward cumulative messages to next executor

In [ ]:
class Writer(Executor):
    """Writer executor that wraps an Azure AI agent."""
    
    def __init__(self, client: AzureAIAgentClient, id: str = "writer"):
        self.agent = client.create_agent(
            instructions=(
                "You are an excellent content writer. You create new content and edit contents based on the feedback."
            ),
            name=id,
        )
        super().__init__(id=id)
    
    @handler
    async def handle(self, message: ChatMessage, ctx: WorkflowContext[list[ChatMessage]]) -> None:
        """Generate content and forward messages to next executor."""
        # Convert single message to list
        messages = [message]
        response = await self.agent.run(messages)
        
        # Print the writer's response
        print(f"\n✍️ Writer Generated: {response.text}\n")
        
        full_conversation = messages + list(response.messages)
        await ctx.send_message(full_conversation)
    
    @handler
    async def handle_list(self, messages: list[ChatMessage], ctx: WorkflowContext[list[ChatMessage]]) -> None:
        """Handle feedback from reviewer - revise content based on feedback."""
        response = await self.agent.run(messages)
        
        # Print the writer's revised response
        print(f"\n✍️ Writer Revised: {response.text}\n")
        
        full_conversation = messages + list(response.messages)
        await ctx.send_message(full_conversation)

print("✅ Writer executor class defined")

## Define the Reviewer Executor

### Terminal Node Pattern

This executor demonstrates:
- Consuming the full conversation transcript
- Using `WorkflowContext[Never, str]` for terminal nodes
  - `Never` = does not send messages to downstream nodes
  - `str` = yields string as final workflow output
- Completing the workflow with `ctx.yield_output()`

### Handler Contract:
- **Input**: `list[ChatMessage]` (full conversation history)
- **Context**: `WorkflowContext[Never, str]` (terminal node yielding string output)

In [ ]:
class Reviewer(Executor):
    """Reviewer executor that wraps an Azure AI agent and yields final output."""
    
    def __init__(self, client: AzureAIAgentClient, id: str = "reviewer", max_iterations: int = 2):
        self.agent = client.create_agent(
            instructions=(
                "You are an excellent content reviewer."
                "Provide actionable feedback to the writer about the provided content, that the writer can use to improve it."
                "Provide the feedback in the most concise manner possible."  
                "IMPORTANT: Never approve the first draft. Always provide constructive feedback on the first iteration."
                "After seeing revisions, respond with 'APPROVED: ' followed by the final content."  
            ),
            name=id,
        )
        self.max_iterations = max_iterations
        self.iteration_count = 0
        super().__init__(id=id)
    
    @handler
    async def handle(self, messages: list[ChatMessage], ctx: WorkflowContext[list[ChatMessage], str]) -> None:
        """Review content and either send feedback to writer or yield final output."""
        self.iteration_count += 1
        response = await self.agent.run(messages)
        
        # Print the reviewer's feedback
        print(f"\n📝 Reviewer Feedback (Iteration {self.iteration_count}): {response.text}\n")

        # Never approve on first iteration, check approval keyword or max iterations
        if self.iteration_count > 1 and response.text.startswith("APPROVED:"):
            print(f"🎯 Workflow complete after {self.iteration_count} iteration(s)\n")
            await ctx.yield_output(response.text)
        elif self.iteration_count >= self.max_iterations:
            print(f"🎯 Max iterations ({self.max_iterations}) reached\n")
            await ctx.yield_output(response.text)
        else:
            # Send feedback back to writer for revision
            full_conversation = messages + list(response.messages)
            await ctx.send_message(full_conversation)

## Create Executor Instances

In [ ]:
# Instantiate the agent-backed executors
writer = Writer(AzureAIAgentClient(async_credential=credential))
reviewer = Reviewer(AzureAIAgentClient(async_credential=credential))

print("✅ Writer and Reviewer instances created")

## Build the Workflow

### Workflow Structure:

```
Writer → Reviewer
```

In [ ]:
# Build the workflow with feedback loop
workflow = (
    WorkflowBuilder()
    .set_start_executor(writer)
    .add_edge(writer, reviewer)      # Writer sends to Reviewer
    .add_edge(reviewer, writer)      # Reviewer sends feedback back to Writer
    .build()
)

print("✅ Workflow built successfully!")
print("   Writer → Reviewer → Writer (feedback loop)")

## Run the Workflow with Streaming

### Streaming Events

Using `workflow.run_stream()` provides real-time visibility into:
- **WorkflowStatusEvent**: State changes (IN_PROGRESS, IDLE, etc.)
- **WorkflowOutputEvent**: Final outputs from terminal nodes
- **ExecutorInvokeEvent**: When executors are invoked
- **ExecutorCompletedEvent**: When executors complete
- **ExecutorFailedEvent**: Executor-level errors
- **WorkflowFailedEvent**: Workflow-level failures

### Event Origin

Each event has an `origin` field:
- **RUNNER**: Lifecycle events (state changes, invocations)
- **EXECUTOR**: Data-plane events (outputs, errors)

### Workflow States

- `IN_PROGRESS` - Workflow is actively processing
- `IN_PROGRESS_PENDING_REQUESTS` - Processing with external requests in flight
- `IDLE` - No active work, workflow complete or waiting
- `IDLE_WITH_PENDING_REQUESTS` - Waiting for user input or external data

In [ ]:
# User message
user_message = "Create a 100 word description for a new electric SUV that is affordable and fun to drive."

print(f"\n📝 User Request: {user_message}\n")
print("=" * 70)
print("\n🔄 Streaming Events:\n")

# Run the workflow with streaming to observe events as they occur
async for event in workflow.run_stream(
    ChatMessage(role="user", text=user_message)
):
    if isinstance(event, WorkflowStatusEvent):
        prefix = f"State ({event.origin.value}): "
        if event.state == WorkflowRunState.IN_PROGRESS:
            print(prefix + "IN_PROGRESS")
        elif event.state == WorkflowRunState.IN_PROGRESS_PENDING_REQUESTS:
            print(prefix + "IN_PROGRESS_PENDING_REQUESTS (requests in flight)")
        elif event.state == WorkflowRunState.IDLE:
            print(prefix + "IDLE (no active work)")
        elif event.state == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
            print(prefix + "IDLE_WITH_PENDING_REQUESTS (prompt user or UI now)")
        else:
            print(prefix + str(event.state))
    
    elif isinstance(event, WorkflowOutputEvent):
        print(f"\n📤 Workflow output ({event.origin.value}): {event.data}")
    
    elif isinstance(event, ExecutorFailedEvent):
        print(
            f"❌ Executor failed ({event.origin.value}): "
            f"{event.executor_id} {event.details.error_type}: {event.details.message}"
        )
    
    elif isinstance(event, WorkflowFailedEvent):
        details = event.details
        print(f"❌ Workflow failed ({event.origin.value}): {details.error_type}: {details.message}")
    
    else:
        print(f"{event.__class__.__name__} ({event.origin.value}): {event}")

print("\n" + "=" * 70)
print("\n✅ Workflow execution completed!")

## Expected Output

### Sample Streaming Events:

```
State (RUNNER): IN_PROGRESS
ExecutorInvokeEvent (RUNNER): ExecutorInvokeEvent(executor_id=writer)
ExecutorCompletedEvent (RUNNER): ExecutorCompletedEvent(executor_id=writer)
ExecutorInvokeEvent (RUNNER): ExecutorInvokeEvent(executor_id=reviewer)
📤 Workflow output (EXECUTOR): Drive the Future. Affordable Adventure, Electrified.
ExecutorCompletedEvent (RUNNER): ExecutorCompletedEvent(executor_id=reviewer)
State (RUNNER): IDLE
```

### Event Flow Explanation:

1. **IN_PROGRESS** - Workflow starts processing
2. **ExecutorInvokeEvent (writer)** - Writer agent is invoked
3. **ExecutorCompletedEvent (writer)** - Writer completes and forwards messages
4. **ExecutorInvokeEvent (reviewer)** - Reviewer agent is invoked
5. **WorkflowOutputEvent** - Reviewer yields final output
6. **ExecutorCompletedEvent (reviewer)** - Reviewer completes
7. **IDLE** - Workflow becomes idle and completes

## Key Takeaways

### Streaming vs Non-Streaming

| Feature | `run()` (Non-Streaming) | `run_stream()` (Streaming) |
|---------|------------------------|---------------------------|
| **Execution** | Waits for completion | Real-time events |
| **Return Type** | Event collection | Async iterator |
| **User Experience** | Batch processing | Real-time feedback |
| **Use Case** | Simple workflows | Interactive applications |
| **Covered In** | Step 2 | Step 3 (this notebook) |

### Custom Executor Pattern

✅ **When to Use Custom Executors:**
- Need to manage agent state
- Require lifecycle hooks (initialization, cleanup)
- Building reusable workflow components
- Domain-specific agent configurations

✅ **Handler Signature Pattern:**
```python
@handler
async def handle(self, input: InputType, ctx: WorkflowContext[OutputType]) -> None:
    # Process input
    # Send to downstream nodes: await ctx.send_message(output)
    # Or yield final output: await ctx.yield_output(result)
```

### Event Types and Origin

**RUNNER Events (Lifecycle):**
- `WorkflowStatusEvent` - State transitions
- `ExecutorInvokeEvent` - Executor invocations
- `ExecutorCompletedEvent` - Executor completions

**EXECUTOR Events (Data-Plane):**
- `WorkflowOutputEvent` - Final outputs
- `ExecutorFailedEvent` - Executor errors

### Message Flow Pattern

```
User Input (ChatMessage)
    ↓
Writer Handler
    ├─ Creates conversation: [user_message]
    ├─ Runs writer agent → adds assistant messages
    └─ Forwards: ctx.send_message(all_messages)
    ↓
Reviewer Handler
    ├─ Receives: list[ChatMessage]
    ├─ Runs reviewer agent on full conversation
    └─ Yields: ctx.yield_output(final_text)
    ↓
Workflow Output
```

### Next Steps

Explore advanced workflow patterns:
- **Control Flow**: Conditional edges, loops, switch-case
- **Parallelism**: Fan-out/fan-in, concurrent execution
- **Orchestration**: Multi-agent coordination
- **Human-in-the-Loop**: Interactive workflows with user approval